# 01 — The exact stack: BBG replication and baselines

Solves the body-variant HJB on BBG's published grid, reproduces the paper's
figure content (quote curves, IV ratios, value surface), and prices the two
baselines (myopic reference, K=36 zero-order hold). Gate numbers and the E1
figure-forensics finding are in the README; this notebook is the visual
companion.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, jax, jax.numpy as jnp
import matplotlib.pyplot as plt
import mfoptmm as mf
from mfoptmm.hjb import make_ops, at_state, interp_axis
from mfoptmm.policy_eval import policy_value, myopic_policy, repeat_block_start
FIGS = ROOT / "docs" / "figures"; FIGS.mkdir(parents=True, exist_ok=True)
print("device:", jax.devices()[0].platform)

In [ ]:
spec = mf.bbg_example(variant="body", profile="replication")
sol = mf.hjb_solve(spec, store_quotes=True)
ops = make_ops(spec)
v00 = float(at_state(sol.v0, ops, 0.0225, 0.0))
print(f"v*(0, nu0, 0) = {v00:,.0f} EUR   E4 min realised diag = {float(sol.min_diag.min()):.3f}")
pv = policy_value(spec, sol.quotes)
print(f"envelope identity |u0 - v0|max = {float(jnp.abs(pv.u0 - sol.v0).max()):.2e}")

In [ ]:
# Figures 4-8 analogue: mid-to-bid / price vs portfolio vega, per strike, t=0, nu0
K = np.asarray(spec.book.strike); Tm = np.asarray(spec.book.mat); O0 = np.asarray(spec.book.O0)
q0 = np.asarray(interp_axis(sol.quotes[0], ops.nu, 0.0225, 0))
gate = np.asarray(ops.gate); V = np.asarray(ops.V)
fig, axes = plt.subplots(1, 5, figsize=(17, 3.1))
for ax, kv in zip(axes, (8, 9, 10, 11, 12)):
    for i in np.where(K == kv)[0]:
        g = gate[:, i, 0]
        ax.plot(V[g]/1e7, q0[g, i, 0]/O0[i], label=f"T={Tm[i]:g}")
    ax.set_title(f"K = {kv}"); ax.set_xlabel(r"$\mathcal{V}^\pi/10^7$")
    ax.legend(fontsize=7)
axes[0].set_ylabel("mid-to-bid / price")
fig.suptitle("Optimal bid quotes at t=0 (paper Figs 4-8 analogue)", y=1.03)
fig.tight_layout(); fig.savefig(FIGS/"nb1_quote_ratios.png", dpi=130); plt.show()

In [ ]:
# Figures 9-13 analogue: IV of the bid / initial IV (K=8 shown; decreasing curves)
fig, ax = plt.subplots(figsize=(4.6, 3.2))
for i in np.where(K == 8)[0]:
    iv0 = mf.implied_vol(O0[i], 10.0, K[i], Tm[i])
    g = gate[:, i, 0]
    ivr = [mf.implied_vol(O0[i] - b, 10.0, K[i], Tm[i]) / iv0 for b in q0[g, i, 0]]
    ax.plot(V[g]/1e7, ivr, label=f"T={Tm[i]:g}")
ax.axhline(1.0, color="k", lw=0.6, ls=":")
ax.set_xlabel(r"$\mathcal{V}^\pi/10^7$"); ax.set_ylabel("IV(bid) / IV0")
ax.set_title("K = 8 (paper Fig 9 analogue)"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(FIGS/"nb1_iv_ratios.png", dpi=130); plt.show()

In [ ]:
# Value surface and the baselines
fig, ax = plt.subplots(figsize=(5.4, 3.4))
c = ax.contourf(V/1e7, np.asarray(ops.nu), np.asarray(sol.v0), levels=24)
fig.colorbar(c, label="v(0, nu, Vpi)  [EUR]")
ax.set_xlabel(r"$\mathcal{V}^\pi/10^7$"); ax.set_ylabel(r"$\nu$")
fig.tight_layout(); fig.savefig(FIGS/"nb1_value.png", dpi=130); plt.show()
u_my = policy_value(spec, myopic_policy(spec))
u_z = policy_value(spec, repeat_block_start(sol.quotes, spec.time_grid.substeps_per_block))
jm = float(at_state(u_my.u0, ops, 0.0225, 0.0)); jz = float(at_state(u_z.u0, ops, 0.0225, 0.0))
print(f"baselines at (nu0, 0):  myopic {jm:,.0f} ({jm/v00:.1%} of v*)   ZOH-36 {jz:,.0f} (costs {(v00-jz)/v00:.2%})")